In [ ]:
!pip install transformers datasets peft trl bitsandbytes accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.6 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset
import pandas as pd

# Public phishing URL dataset — 11,000+ labeled examples
raw = load_dataset("shawhin/phishing-site-classification", split="train")

# See what it looks like
print(raw[0])
print(f"\nTotal examples: {len(raw)}")
print(f"Column names: {raw.column_names}")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/98.0k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/21.4k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/24.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2100 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/450 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/450 [00:00<?, ? examples/s]

{'text': "http://bazurashop.com/idex.html?sfm_from_iframe=1',300,350", 'labels': 1}

Total examples: 2100
Column names: ['text', 'labels']


In [ ]:
# Check what the dataset actually looks like
print(raw.column_names)
print(raw[0])

['text', 'labels']
{'text': "http://bazurashop.com/idex.html?sfm_from_iframe=1',300,350", 'labels': 1}


In [ ]:
import random
random.seed(42)

# This dataset uses "text" for URL and "labels" for the label
phishing = [x for x in raw if x["labels"] == 1]
safe     = [x for x in raw if x["labels"] == 0]

print(f"Total phishing: {len(phishing)}")
print(f"Total safe:     {len(safe)}")

# Sample 60 of each for a balanced dataset
sample_phishing = random.sample(phishing, min(60, len(phishing)))
sample_safe     = random.sample(safe,     min(60, len(safe)))
combined        = sample_phishing + sample_safe
random.shuffle(combined)

# Format — note: URL is in "text" column, not "url"
def format_example(ex):
    url   = ex["text"]    # ← "text" is the URL in this dataset
    label = "UNSAFE" if ex["labels"] == 1 else "SAFE"
    if label == "UNSAFE":
        completion = f"UNSAFE. This URL shows signs of phishing: {url}"
    else:
        completion = f"SAFE. This appears to be a legitimate URL: {url}"
    return {
        "text": f"### Instruction:\nAnalyze this URL for phishing: {url}\n\n### Response:\n{completion}"
    }

from datasets import Dataset
dataset = Dataset.from_list([format_example(e) for e in combined])

print(f"\nDataset ready: {len(dataset)} examples")
print("\n--- Preview ---")
print(dataset[0]["text"])

# Also save these for the evaluation step later
url_col   = "text"
label_col = "labels"
phishing_val = 1
safe_val     = 0

Total phishing: 1046
Total safe:     1054

Dataset ready: 120 examples

--- Preview ---
### Instruction:
Analyze this URL for phishing: ilansorgula.com/h5io1/aoil/lokiJh1/da1ce0fa6edba4c24b4cc2bc05d7b559/

### Response:
UNSAFE. This URL shows signs of phishing: ilansorgula.com/h5io1/aoil/lokiJh1/da1ce0fa6edba4c24b4cc2bc05d7b559/


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
MODEL = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb_config,
    token=hf_token, device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(MODEL, token=hf_token)
tokenizer.pad_token = tokenizer.eos_token

lora_config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

training_args = SFTConfig(
    output_dir="./phishing-lora-v2",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    logging_steps=5,   # log every 5 steps (less noise with more data)
    max_length=256,
    dataset_text_field="text"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
)
trainer.train()

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

trainable params: 1,572,864 || all params: 1,712,949,248 || trainable%: 0.0918


Adding EOS to train dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

Step,Training Loss
5,2.774803
10,2.756880
15,2.246725
20,2.266766
25,1.862506
30,1.595807
35,1.338070
40,1.550156
45,1.421736
50,1.440250


TrainOutput(global_step=180, training_loss=1.4390781270133124, metrics={'train_runtime': 219.1051, 'train_samples_per_second': 1.643, 'train_steps_per_second': 0.822, 'total_flos': 343474950070272.0, 'train_loss': 1.4390781270133124})

In [ ]:
import torch

test_urls = [
    ("chase-bank-secure-login.info/account/update", "UNSAFE"),
    ("amazon.com/orders/return",                   "SAFE"),   # the one that failed!
    ("dropbox-file-shared.ru/download/invoice.pdf", "UNSAFE"),
]

model.eval()
correct = 0

for url, expected in test_urls:
    prompt = f"### Instruction:\nAnalyze this URL for phishing: {url}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=False,
            repetition_penalty=1.3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    response  = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    got_right = expected in response.upper()
    if got_right: correct += 1

    status = "✓" if got_right else "✗"
    print(f"{status} [{expected}] {url}")
    print(f"   → {response}\n")

print(f"Score: {correct}/3")

✓ [UNSAFE] chase-bank-secure-login.info/account/update
   → UNSAFE. This is a UNSAFE URL: Chase Bank Secure Login / account / update

✓ [SAFE] amazon.com/orders/return
   → SAFE. This appears to be a legitimate Amazon order return page."

✗ [UNSAFE] dropbox-file-shared.ru/download/invoice.pdf
   → SAFE. This appears to be a legitimate URL: dropbox-file-shared.ru/download/invoice.pdf

Score: 2/3


In [ ]:
import torch

# Hold-out test set — examples not seen during training
test_raw_phishing = random.sample([x for x in phishing if x not in sample_phishing], 10)
test_raw_safe     = random.sample([x for x in safe     if x not in sample_safe],     10)
test_examples     = test_raw_phishing + test_raw_safe
random.shuffle(test_examples)

correct = 0
model.eval()

for ex in test_examples:
    url      = ex["text"]    # ← fixed
    expected = "UNSAFE" if ex["labels"] == 1 else "SAFE"   # ← fixed
    prompt   = f"### Instruction:\nAnalyze this URL for phishing: {url}\n\n### Response:\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=30,
            do_sample=False, repetition_penalty=1.3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    input_len = inputs["input_ids"].shape[1]
    response  = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    got_right = expected in response.upper()
    if got_right: correct += 1

    status = "✓" if got_right else "✗"
    print(f"{status} [{expected}] {url[:60]}")

print(f"\nFinal accuracy: {correct}/20 = {correct/20*100:.0f}%")

✓ [UNSAFE] paypal.com-us.cgi-bin-webscr-cmd.login-submit-dispatch.63249
✓ [SAFE] temp.totalcapitol.com/?bill_id=201120120AB1136
✗ [UNSAFE] picass0.com/rtl/sign.php
✓ [SAFE] heroesofcapitalism.blogspot.com/2009/04/ewing-marion-kauffma
✓ [SAFE] 2010games.nytimes.com/events/freestyle-skiing/womens-moguls/
✗ [UNSAFE] ralucatodorut.com/wp-includes/mmc.htm
✓ [UNSAFE] www.sunjsc.vn/admin/webroot/video/upload/hmrc/allaccounts/me
✓ [SAFE] urbanspoon.com/n/2/40588/Chicago/Bartlett-IL-restaurants
✓ [UNSAFE] krosha.myjino.ru/pggzm
✓ [SAFE] thecanadianencyclopedia.com/index.cfm?PgNm=TCE&Params=U1ARTU
✓ [UNSAFE] 89.108.83.45/apache_handler.php
✓ [UNSAFE] eu.diablo.net.ms.sy-login.in/login.html?&amp;amp;amp;amp;amp
✓ [SAFE] freespace.virgin.net/lcba.ts/
✗ [UNSAFE] astra-antiques.com/bt32u5
✓ [SAFE] topix.net/member/editor-application?node=city/kennett-mo
✓ [SAFE] absoluteastronomy.com/topics/List_of_United_States_Coast_Gua
✓ [UNSAFE] vfqqgr1971.cba.pl
✓ [SAFE] imdb.com/name/nm0272857/
✓ [UNSAFE] stbx